# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset is sourced via the Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets and their IDs.

The `mlcroissant.Dataset` object provides access to the record sets defined in the dataset's Croissant schema.

Let's examine which record sets are available, along with their `@id`s and fields.

In [ ]:
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"  Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, type: {f.data_type})")
    print("")

# For demonstration, pick the first record set
if record_sets:
    demo_record_set = record_sets[0]
    print(f"Sample records from Record Set '{demo_record_set.name}' (@id: {demo_record_set.id}):")
    for i, x in enumerate(dataset.records(record_set=demo_record_set.id)):
        print(x)
        if i > 3:
            break

## 3. Data Extraction
Load data from record sets into DataFrames for analysis.

We use the `@id`s of each record set and their fields to load the data.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set @id: {record_set_id}")

# Show columns from the demonstration record set
demo_id = record_set_ids[0] if record_set_ids else None
if demo_id:
    print(f"Columns for record set @id: {demo_id}")
    print(dataframes[demo_id].columns.tolist())
    print(dataframes[demo_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes for further analysis.

For demonstration, let's select a field (column) from the main record set. Ensure you reference fields or columns by their `@id`.

In [ ]:
# Example: Analyze numeric mental health assessment scores

# Identify candidate numeric fields by @id
demo_rs = record_sets[0] if record_sets else None
if not demo_rs:
    raise ValueError("No record sets defined in dataset.")

# Example candidate field (adjust as appropriate)
numeric_field_id = None
for field in demo_rs.fields:
    if field.data_type in ["schema:Float", "schema:Integer"] and ("score" in field.name.lower() or "GAD-7" in field.name or "PHQ-9" in field.name or "PSQ" in field.name):
        numeric_field_id = field.id
        break

# Fallback to another numeric field if not found
if not numeric_field_id:
    # Try first integer/float field
    for field in demo_rs.fields:
        if field.data_type in ["schema:Float", "schema:Integer"]:
            numeric_field_id = field.id
            break

print(f"Numeric field @id selected: {numeric_field_id}")

df = dataframes[demo_rs.id]
threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns else 10

# Filter records where score exceeds threshold
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a key demographic field
    group_field_id = None
    for field in demo_rs.fields:
        if field.data_type == "schema:Text" and field.name.lower() in ("gender", "village", "marital status", "level_of_education"):
            group_field_id = field.id
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in dataframe columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we demonstrate a histogram of a key numeric assessment score and a boxplot grouped by a demographic variable (using the `@id` for all references).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`.

- Loaded dataset metadata and content using the Croissant schema.
- Identified available record sets and fields by their `@id`.
- Performed basic extraction of tabular data using record set and field IDs.
- Filtered and normalized numeric assessment scores, and grouped by demographic variables.
- Visualized data distributions and group differences.

This approach demonstrates reproducible FAIR-ML dataset exploration, leveraging the standardized structure of Croissant schema and the flexibility of `mlcroissant`.